# RecapAI - Meeting Summarizer
**Course:** CSC 603 - Generative AI | **Team:** Adrian Aquino, Charlie Huynh, Will Brust | **Spring 2026**

## Install Libraries

In [ ]:
!pip install -q transformers accelerate huggingface_hub

print('Done!')

Done!


## Import Libraries

In [ ]:
import json
import torch
from transformers import pipeline
from huggingface_hub import login
from google.colab import files

print('Done!')

Done!


## Hugging Face Login

In [ ]:
RecapAI = 'YOUR_HF_TOKEN_HERE'  # paste your Hugging Face token here

login(token=RecapAI)

print('Logged in!')

## Load the Model

In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

# use GPU if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

llm = pipeline(
    task='text-generation',
    model=MODEL_NAME,
    token=RecapAI,
    device_map='auto',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)

print('Model loaded!')

Using: cuda


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded!


## System Prompt

In [ ]:
# role: what the model is
ROLE = 'You summarize meeting transcripts. You only use information from the transcript and never add anything that was not said.'

# task: what the model must do and how to format the output
TASK = """Read the transcript and return only a JSON object with these 4 fields:
- summary: a short paragraph summarizing the meeting
- decisions: a list of decisions that were made
- assigned_tasks: a list of objects with owner, task, and due_date (use 'Not specified' if no date was mentioned)
- open_questions: a list of questions that were raised but not answered

If a section has nothing to report, return an empty list. Do not include any text outside the JSON."""

print('Prompt ready!')

Prompt ready!


## Summarize Function

In [ ]:
ROLE = "You are an AI assistant that summarizes meeting transcripts into structured JSON."

TASK = """Summarize the following meeting transcript.
Return ONLY a JSON object with exactly these 4 keys:
- "summary": a short 2-3 sentence overview of the meeting
- "decisions": a list of decisions that were made
- "assigned_tasks": a list of objects with "who", "what", and "due" fields
- "open_questions": a list of questions that were raised but not resolved

Do not include any explanation or text outside the JSON."""

def summarize_meeting(transcript, max_tokens=1024):
    """Sends a transcript to the LLM and returns structured JSON.

    Args:
        transcript (str): The raw meeting transcript.
        max_tokens (int): Max tokens to generate. Default is 1024.

    Returns:
        dict: JSON output with summary, decisions, assigned_tasks, open_questions.
    """

    messages = [
        {'role': 'system', 'content': f'{ROLE}'},
        {'role': 'user',   'content': f'{TASK}\n\nTranscript:\n{transcript}'}
    ]

    print('Sending to model...')
    response = llm(
        messages,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=1.0
    )

    # pull text from response
    raw_output = response[0]['generated_text'][-1]['content']
    print('Response received. Parsing...')

    return try_parse_json(raw_output)

print('Function defined!')

Function defined!


## JSON Parser with Fallback

In [ ]:
def try_parse_json(raw_output):
    """Parses model output as JSON. Retries once on failure, then falls back to raw text.

    Args:
        raw_output (str): Raw text returned by the model.

    Returns:
        dict: Parsed JSON or a fallback dict with raw text under 'summary'.
    """

    # attempt 1: direct parse
    try:
        start  = raw_output.index('{')
        end    = raw_output.rindex('}') + 1
        parsed = json.loads(raw_output[start:end])
        print('Parsed on first attempt.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('First attempt failed. Trying repair...')

    # attempt 2: ask model to fix its own output
    try:
        repair_messages = [
            {'role': 'system', 'content': 'You fix broken JSON. Return only valid JSON and nothing else.'},
            {'role': 'user',   'content': f'Fix this JSON:\n\n{raw_output}'}
        ]
        repair_response = llm(repair_messages, max_new_tokens=1024, do_sample=False, temperature=1.0)
        repaired        = repair_response[0]['generated_text'][-1]['content']
        start           = repaired.index('{')
        end             = repaired.rindex('}') + 1
        parsed          = json.loads(repaired[start:end])
        print('Parsed after repair.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('Repair failed. Returning fallback.')

    # fallback: return raw text so something is shown instead of a crash
    return {
        'summary': raw_output,
        'decisions': [],
        'assigned_tasks': [],
        'open_questions': ['[Could not parse output - see summary for raw text]']
    }

print('Function defined!')

Function defined!


## Validate Output

In [ ]:
def validate_output(result):
    """Checks that all 4 required sections exist in the output.

    Args:
        result (dict): Parsed output from summarize_meeting().

    Returns:
        bool: True if all sections present, False if any are missing.
    """

    required    = ['summary', 'decisions', 'assigned_tasks', 'open_questions']
    all_present = True

    for section in required:
        if section not in result:
            print(f"WARNING: Missing -> '{section}'")
            all_present = False
        else:
            print(f"OK: '{section}'")

    print('\nAll good!' if all_present else '\nSome sections missing. Check the prompt.')
    return all_present

print('Function defined!')

Function defined!


## Input Transcript
Paste your transcript below or upload a .txt file.

In [ ]:
# option 1: type or paste your transcript when prompted
print('Paste your transcript below, then press Enter twice when done.')
TRANSCRIPT = input('Transcript: ')

print(f'Transcript set! Length: {len(TRANSCRIPT)} characters')

Paste your transcript below, then press Enter twice when done.
Transcript: Alright, I think everyone's here now, so let's just get started.  Yeah, sorry, I was having trouble with my mic for a second there.  No worries. So the main thing I wanted to cover today is where we're at with the Q2 rollout. Last I heard from Derek, the backend was still waiting on the API changes, is that still the case?  Yeah, so, um, we actually got a lot of that unblocked on Friday. The auth endpoints are done, I just need to finish writing the tests and then I can hand it off to QA.  Okay that's good. That's better than I thought actually.  Wait, are the tests blocking the handoff or can QA start on the happy path stuff while you finish them?  I mean, technically they could start, but I'd rather not because last time we did that we ended up finding stuff that was already caught by the unit tests and it was just like, a waste of everyone's time.  Fair enough.  What's the timeline looking like then? Because 

In [ ]:
# option 2: upload a .txt file
# run this cell and use the file picker to select your .txt file
from google.colab import files
uploaded = files.upload()

# reads the first uploaded file
filename  = list(uploaded.keys())[0]
TRANSCRIPT = uploaded[filename].decode('utf-8')

print(f'Loaded: {filename}')
print(f'Length: {len(TRANSCRIPT)} characters')

IndexError: list index out of range

## Run

In [ ]:
# stop if no transcript
if TRANSCRIPT.strip() == '' or TRANSCRIPT.strip() == 'Paste your transcript here.':
    print('ERROR: No transcript provided. Paste your transcript in the Input cell first.')
else:
    result = summarize_meeting(TRANSCRIPT)

    # save output as JSON file
    with open('output.json', 'w') as f:
        json.dump(result, f, indent=2)

    print('Done! Output saved to output.json')

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sending to model...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response received. Parsing...
First attempt failed. Trying repair...
Parsed after repair.
Done! Output saved to output.json
